# Enhanced RAG Pipeline with Hybrid Search
## Isolated Notebook for Retrieval and Querying

This notebook implements an enhanced RAG pipeline with hybrid search (BM25 + Semantic) for dermatology documents.

**Key Features:**
- Hybrid search combining keyword (BM25) and semantic (embedding) retrieval
- Easy switching between recursive and agentic chunking vector stores
- Configurable retrieval parameters
- Comparison tools for hybrid vs semantic-only search

**Note:** This notebook loads existing vector stores from `./chroma_db`. It does not rebuild embeddings.


## 1. Install Required Dependencies

In [ ]:
# Install required packages for hybrid search
%pip install rank-bm25

In [ ]:
%pip install langchain langchain-core langchain-community langchain-google-genai langchain-openai

In [ ]:
%pip install langchain-classic

In [ ]:
%pip install chromadb

## 2. Import Libraries


In [ ]:
import os
import getpass
from pathlib import Path
from typing import List, Dict, Optional

# LangChain imports
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document

print("All libraries imported successfully")


## 3. Configuration


In [ ]:
# ============================================================================
# CONFIGURATION - Adjust these parameters as needed
# ============================================================================

# Vector store configuration
VECTOR_STORE_PATH = "./chroma_db"
CHUNKING_METHOD = "recursive"  # Options: "recursive" or "agentic"

# Hybrid search weights [BM25_weight, Semantic_weight]
# For medical domain, balanced weights work well
# Tune based on your evaluation results
HYBRID_WEIGHTS = [0.4, 0.6]  # 40% BM25, 60% Semantic

# Retrieval parameters
RETRIEVAL_K = 5  # Number of documents to retrieve from each retriever

# Weight configurations for testing
WEIGHT_CONFIGS = {
    "balanced": [0.4, 0.6],      # 40% BM25, 60% Semantic (recommended)
    "keyword_heavy": [0.6, 0.4],  # 60% BM25, 40% Semantic (good for exact terms)
    "semantic_heavy": [0.3, 0.7], # 30% BM25, 70% Semantic (good for concepts)
    "equal": [0.5, 0.5]           # 50/50 split
}

print("="*80)
print("CONFIGURATION")
print("="*80)
print(f"Chunking Method: {CHUNKING_METHOD}")
print(f"Hybrid Weights: BM25={HYBRID_WEIGHTS[0]}, Semantic={HYBRID_WEIGHTS[1]}")
print(f"Retrieval K: {RETRIEVAL_K}")
print(f"Vector Store Path: {VECTOR_STORE_PATH}")
print("="*80)


## 4. Setup Environment

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted")

In [ ]:
# Define the path to the vector store in Google Drive
# IMPORTANT: Update this path to where your 'chroma_db' folder is located in your Drive
VECTOR_STORE_PATH = "/content/drive/MyDrive/chroma_db" # Adjust this path as needed
print(f"Using vector store path: {VECTOR_STORE_PATH}")

In [ ]:
import os

# Set up Google API key for Gemini Pro
api_key = getpass.getpass("Enter your Google API Key: ")
if not api_key or not api_key.strip():
    raise ValueError("Google API key is required. Please provide a valid API key.")
os.environ["GOOGLE_API_KEY"] = api_key.strip()

# Validate API key format (basic check)
api_key = os.environ.get("GOOGLE_API_KEY", "")
if len(api_key) < 20:  # Basic validation - Google API keys are typically longer
    raise ValueError(
        "Invalid API key format detected. "
        "Please ensure you've entered a valid Google API key."
    )

print("Google API key configured")

In [ ]:
%pip install huggingface-hub

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# Initialize embeddings model (same as used for vector store)
print("Initializing embeddings model...")

embeddings = HuggingFaceEmbeddings(
    model_name="google/embeddinggemma-300m",
    model_kwargs={'device': 'cuda'},  # Use 'cuda' if GPU available
    encode_kwargs={'normalize_embeddings': True}  # Normalize for cosine similarity
)

print("Embeddings model initialized")

## 5. Load Vector Store with Easy Switching


In [ ]:
def load_vectorstore(chunking_method: str = "recursive") -> Chroma:
    """
    Load vector store based on chunking method.

    Args:
        chunking_method: "recursive" or "agentic"

    Returns:
        Chroma vectorstore instance

    Raises:
        ValueError: If collection doesn't exist or is empty
        FileNotFoundError: If vector store path doesn't exist
    """
    collection_name = {
        "recursive": "dermatology_docs_recursive",
        "agentic": "dermatology_docs_agentic"
    }.get(chunking_method.lower(), "dermatology_docs_recursive")

    print(f"Loading vector store: {collection_name}...")

    # Check if vector store path exists
    if not os.path.exists(VECTOR_STORE_PATH):
        raise FileNotFoundError(
            f"Vector store path not found: {VECTOR_STORE_PATH}\n"
            "Please ensure the vector store has been created first."
        )

    try:
        vectorstore = Chroma(
            persist_directory=VECTOR_STORE_PATH,
            embedding_function=embeddings,
            collection_name=collection_name
        )

        # Get collection info
        collection = vectorstore._collection
        doc_count = collection.count()

        if doc_count == 0:
            raise ValueError(
                f"Collection '{collection_name}' exists but is empty.\n"
                "Please ensure the vector store has been populated with documents."
            )

        print(f"Vector store loaded successfully!")
        print(f"  Collection: {collection_name}")
        print(f"  Documents: {doc_count}")

        return vectorstore

    except Exception as e:
        if "does not exist" in str(e).lower() or "not found" in str(e).lower():
            raise ValueError(
                f"Collection '{collection_name}' not found in vector store.\n"
                f"Available collections may need to be created first.\n"
                f"Original error: {e}"
            )
        raise

# Load the vector store based on configuration
vectorstore = load_vectorstore('agentic')

## 6. Load Chunks from Vector Store

In [ ]:
def load_chunks_from_vectorstore(vectorstore: Chroma) -> List[Document]:
    """
    Load all chunks from vector store for BM25 indexing.

    Args:
        vectorstore: Chroma vectorstore instance

    Returns:
        List of Document objects

    Raises:
        ValueError: If no documents are found in the vector store
    """
    print("Loading chunks from vector store...")

    try:
        # Get all documents from the collection
        collection = vectorstore._collection
        all_results = collection.get()

        # Extract documents
        documents = []
        ids = all_results.get("ids", [])
        metadatas = all_results.get("metadatas", [])
        documents_text = all_results.get("documents", [])

        total = len(ids)

        if total == 0:
            raise ValueError(
                "No documents found in vector store. "
                "Please ensure the vector store has been populated."
            )

        print(f"Found {total} chunks in vector store")

        # Validate that all lists have the same length
        if not (len(ids) == len(metadatas) == len(documents_text)):
            raise ValueError(
                f"Mismatch in vector store data: "
                f"ids={len(ids)}, metadatas={len(metadatas)}, documents={len(documents_text)}"
            )

        # Create Document objects
        for i, (doc_id, text, metadata) in enumerate(zip(ids, documents_text, metadatas)):
            if (i + 1) % 1000 == 0:
                print(f"  Processing chunk {i + 1}/{total}...")

            doc = Document(
                page_content=text,
                metadata=metadata or {}
            )
            documents.append(doc)

        print(f"Loaded {len(documents)} chunks")
        return documents

    except Exception as e:
        if isinstance(e, ValueError):
            raise
        raise ValueError(f"Error loading chunks from vector store: {e}")

# Load chunks for BM25 indexing
chunks = load_chunks_from_vectorstore(vectorstore)

## 7. Build BM25 Index

In [ ]:
print("Building BM25 index...")
print(f"Processing {len(chunks)} chunks...")

# Create BM25 retriever from chunks
# BM25 is keyword-based and works great for medical terms
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = RETRIEVAL_K  # Retrieve top k from BM25

print("BM25 index built successfully")
print(f"  Index size: {len(chunks)} documents")
print(f"  Retrieval k: {bm25_retriever.k}")

## 8. Create Semantic Retriever


In [ ]:
# Create semantic retriever from existing Chroma vector store
semantic_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": RETRIEVAL_K}  # Retrieve top k from semantic search
)

print("Semantic retriever created")
print(f"  Retrieval k: {RETRIEVAL_K}")

## 9. Hybrid Search Implementation


In [ ]:
# Create Ensemble Retriever combining BM25 and semantic retrievers
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=HYBRID_WEIGHTS
)

print("Hybrid retriever created")
print(f"  Weights: BM25={HYBRID_WEIGHTS[0]}, Semantic={HYBRID_WEIGHTS[1]}")
print(f"  Total retrievers: 2 (BM25 + Semantic)")

In [ ]:
def test_hybrid_weights(question: str, weights: list, show_docs: bool = False):
    """
    Test hybrid search with different weights.

    Args:
        question: Test question
        weights: [BM25_weight, Semantic_weight]
        show_docs: Whether to display retrieved documents
    """
    test_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, semantic_retriever],
        weights=weights
    )

    # Get results from both retrievers
    bm25_docs = bm25_retriever.invoke(question)
    semantic_docs = semantic_retriever.invoke(question)
    hybrid_docs = test_retriever.invoke(question)

    print(f"\n{'='*80}")
    print(f"Weights: BM25={weights[0]}, Semantic={weights[1]}")
    print(f"{'='*80}")
    print(f"BM25 retrieved: {len(bm25_docs)} docs")
    print(f"Semantic retrieved: {len(semantic_docs)} docs")
    print(f"Hybrid retrieved: {len(hybrid_docs)} docs")

    if show_docs:
        print(f"\nHybrid retrieved documents:")
        for i, doc in enumerate(hybrid_docs[:3], 1):
            category = doc.metadata.get("category", "Unknown")
            print(f"  [{i}] Category: {category}")
            print(f"      Preview: {doc.page_content[:150]}...")

    return hybrid_docs

print("Weight testing function created")

## 10. Initialize LLM

In [ ]:
# Initialize Gemini Pro
print("Initializing Gemini Pro model...")

try:
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-pro",
        temperature=0.1,  # Low temperature for factual medical responses
        convert_system_message_to_human=True
    )
    print("Gemini Pro initialized")

except Exception as e:
    error_msg = str(e).lower()
    if "api key" in error_msg or "authentication" in error_msg:
        raise ValueError(
            "Failed to initialize Gemini Pro: Invalid or missing API key.\n"
            "Please check your GOOGLE_API_KEY environment variable."
        ) from e
    elif "quota" in error_msg or "limit" in error_msg:
        raise ValueError(
            "Failed to initialize Gemini Pro: API quota exceeded or rate limited.\n"
            "Please check your Google Cloud API quotas."
        ) from e
    else:
        raise ValueError(
            f"Failed to initialize Gemini Pro: {e}\n"
            "Please check your API key and network connection."
        ) from e

In [ ]:
# Create a medical-focused prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a medical expert assistant specializing in dermatology. Use the following pieces of context from medical literature to answer the question.

Guidelines:
- Provide accurate, evidence-based answers from the context
- If you don't know the answer from the context, say so - don't make up information
- Include relevant details like treatment options, dosages, and guidelines when available
- Cite the source category (e.g., Acne, Eczema, Psoriasis) when referencing information
- Use clear, professional medical language

Context:
{context}"""),
    ("human", "{question}")
])

print("Custom prompt template created")

## 11. Create Enhanced RAG Chain


In [ ]:
# Helper function to format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create the hybrid RAG chain using LCEL
rag_chain_hybrid = (
    {
        "context": hybrid_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("Hybrid RAG chain created successfully!")

# Also create semantic-only chain for comparison
rag_chain_semantic = (
    {
        "context": semantic_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("Semantic-only RAG chain created for comparison")

## 12. Query Functions


In [ ]:
def query_rag_hybrid(
    question: str,
    show_sources: bool = True,
    show_breakdown: bool = False,
    compare_with_semantic: bool = False
) -> dict:
    """
    Query the hybrid RAG pipeline.

    Args:
        question: The medical question to ask
        show_sources: Whether to display source documents
        show_breakdown: Whether to show BM25 vs Semantic retrieval breakdown
        compare_with_semantic: Whether to compare with semantic-only retrieval

    Returns:
        Dictionary with answer and source documents
    """
    print(f"\n{'='*80}")
    print(f"QUESTION: {question}")
    print("="*80)

    # Show breakdown if requested
    if show_breakdown:
        bm25_docs = bm25_retriever.invoke(question)
        semantic_docs = semantic_retriever.invoke(question)
        hybrid_docs = hybrid_retriever.invoke(question)

        print(f"\nRetrieval Breakdown:")
        print(f"   BM25 (keyword): {len(bm25_docs)} documents")
        print(f"   Semantic (embedding): {len(semantic_docs)} documents")
        print(f"   Hybrid (combined): {len(hybrid_docs)} documents")

        # Show overlap - compare by exact content string (same as EnsembleRetriever does)
        bm25_contents = {doc.page_content for doc in bm25_docs}
        semantic_contents = {doc.page_content for doc in semantic_docs}
        overlap = len(bm25_contents & semantic_contents)
        print(f"   Overlap: {overlap} documents")

        # Verify the math matches
        expected_unique = len(bm25_docs) + len(semantic_docs) - overlap
        print(f"   Expected unique: {expected_unique} documents")
        if expected_unique != len(hybrid_docs):
            print(f"   Note: EnsembleRetriever returned {len(hybrid_docs)} documents")


    # Query the hybrid chain
    print("\nQuerying hybrid RAG chain...")
    answer = rag_chain_hybrid.invoke(question)

    # Display answer
    print("\n" + "="*80)
    print("ANSWER (Hybrid Search):")
    print("="*80)
    print(answer)

    # Compare with semantic-only if requested
    if compare_with_semantic:
        print("\n" + "="*80)
        print("COMPARISON: Semantic-Only Search")
        print("="*80)
        semantic_answer = rag_chain_semantic.invoke(question)
        print(semantic_answer)

    # Retrieve source documents
    source_docs = None
    if show_sources:
        source_docs = hybrid_retriever.invoke(question)
        print("\n" + "="*80)
        print("SOURCES (from hybrid search):")
        print("="*80)
        for i, doc in enumerate(source_docs, 1):
            category = doc.metadata.get("category", "Unknown")
            source_file = doc.metadata.get("source_file", "Unknown")
            page = doc.metadata.get("page", "?")
            print(f"\n[{i}] Category: {category} | File: {source_file} | Page: {page}")
            print(f"    Content preview: {doc.page_content[:200]}...")

    print("\n" + "="*80)
    return {"answer": answer, "source_documents": source_docs}

## 13. Example Queries

In [ ]:
# Example 1: Query with breakdown
result = query_rag_hybrid(
    "What are the treatment options for eczema?",
    show_sources=True,
    show_breakdown=True
)


In [ ]:
# Example 2: Compare hybrid vs semantic-only
result = query_rag_hybrid(
    "How to treat psoriasis?",
    show_sources=True,
    compare_with_semantic=True
)


In [ ]:
# Example 3: Test different weight configurations
test_question = "What causes urticaria?"

print("Testing different weight configurations:")
print("="*80)

for name, weights in WEIGHT_CONFIGS.items():
    print(f"\n{name.upper()}:")
    test_hybrid_weights(test_question, weights, show_docs=False)


## 14. Switch Vector Store (Recursive - Agentic)

To switch between recursive and agentic chunking vector stores, update the `CHUNKING_METHOD` in the configuration section (Section 3) and re-run the cells from Section 5 onwards.

In [ ]:
# Example: Switch to agentic chunking
# Uncomment and run the cells below to switch vector stores

# CHUNKING_METHOD = "agentic"  # Change from "recursive" to "agentic"
# vectorstore = load_vectorstore(CHUNKING_METHOD)
# chunks = load_chunks_from_vectorstore(vectorstore)
# bm25_retriever = BM25Retriever.from_documents(chunks)
# bm25_retriever.k = RETRIEVAL_K
# semantic_retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})
# hybrid_retriever = EnsembleRetriever(
#     retrievers=[bm25_retriever, semantic_retriever],
#     weights=HYBRID_WEIGHTS
# )
# # Recreate RAG chain with new retriever
# rag_chain_hybrid = (
#     {
#         "context": hybrid_retriever | format_docs,
#         "question": RunnablePassthrough()
#     }
#     | prompt
#     | llm
#     | StrOutputParser()
# )

print("To switch vector stores, update CHUNKING_METHOD in Section 3 and re-run from Section 5")


## 15. RAGAS Evaluation


In [ ]:
# Install RAGAS and pandas for evaluation
%pip install ragas pandas datasets

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)
from ragas.run_config import RunConfig
from datasets import Dataset
import pandas as pd
import time # Import the time module

def evaluate_rag_pipeline(
    test_questions: list,
    ground_truths: list = None,
    use_hybrid: bool = True
):
    """
    Evaluate RAG pipeline using RAGAS metrics.

    Args:
        test_questions: List of test questions
        ground_truths: Optional list of expected answers for comparison
        use_hybrid: Whether to use hybrid retriever (True) or semantic-only (False)

    Returns:
        Tuple of (evaluation_result, results_dict)
    """
    results = []

    # Choose retriever based on use_hybrid flag
    retriever_to_use = hybrid_retriever if use_hybrid else semantic_retriever
    chain_to_use = rag_chain_hybrid if use_hybrid else rag_chain_semantic

    print(f"Evaluating {'hybrid' if use_hybrid else 'semantic-only'} RAG pipeline...")
    print(f"Number of test questions: {len(test_questions)}")
    print("="*80)

    for i, question in enumerate(test_questions):
        print(f"Processing question {i+1}/{len(test_questions)}: {question[:60]}...")

        # Get answer and context
        answer = chain_to_use.invoke(question)
        contexts = retriever_to_use.invoke(question)
        context_list = [doc.page_content for doc in contexts]

        result = {
            "question": question,
            "answer": answer,
            "contexts": context_list,
        }

        if ground_truths and i < len(ground_truths):
            result["ground_truth"] = ground_truths[i]

        results.append(result)

    print("\n" + "="*80)
    print("Converting to dataset format...")

    # Convert to dataset format
    dataset = Dataset.from_list(results)

    # Evaluate with RAGAS metrics
    print("Running RAGAS evaluation...")
    metrics = [faithfulness, answer_relevancy]

    if ground_truths:
        metrics.extend([context_precision, context_recall])

    run_config = RunConfig(max_workers=1, timeout=60)

    evaluation_result = evaluate(
        dataset,
        metrics=metrics,
        llm=llm,
        embeddings=embeddings,
        run_config = run_config
    )

    print("="*80)
    print("Evaluation complete!")
    print("="*80)

    return evaluation_result, results

print("RAGAS evaluation function created")

In [ ]:
# Example: Evaluate RAG pipeline with test questions
# You can customize these questions based on your needs

# test_questions = [
#     "How to treat psoriasis?",
#     "What are the treatment options for eczema?",
#     "What causes urticaria?",
#     "What is the first-line treatment for acne?",
#     "How is rosacea managed?"
# ]

test_questions = [
    "What are the treatment options for eczema?",
]

# Optional: Provide ground truth answers for more comprehensive evaluation
# ground_truths = [
#     "Psoriasis treatment includes topical treatments, phototherapy, systemic medications, and biologics...",
#     "Eczema treatment involves moisturizers, topical corticosteroids, and avoiding triggers...",
#     # ... add more ground truths
# ]

# Evaluate hybrid RAG pipeline
print("Evaluating Hybrid RAG Pipeline:")
print("="*80)
eval_result_hybrid, results_hybrid = evaluate_rag_pipeline(
    test_questions=test_questions,
    ground_truths=None,  # Set to ground_truths list if available
    use_hybrid=True
)

print("\nEvaluation Results (Hybrid):")
print(eval_result_hybrid)


In [ ]:
# Optional: Compare hybrid vs semantic-only retrieval
# Uncomment to run comparison evaluation

print("\n" + "="*80)
print("Evaluating Semantic-Only RAG Pipeline (for comparison):")
print("="*80)
eval_result_semantic, results_semantic = evaluate_rag_pipeline(
    test_questions=test_questions,
    ground_truths=None,
    use_hybrid=True
)

print("\nEvaluation Results (Semantic-Only):")
print(eval_result_semantic)

print("\n" + "="*80)
print("COMPARISON SUMMARY:")
print("="*80)
print("Hybrid Search Metrics:")
print(eval_result_hybrid)
print("\nSemantic-Only Search Metrics:")
print(eval_result_semantic)
